In [1]:
import pandas as pd
import unicodedata
import spacy
import sys

sys.path.append("../src")
from normalization import normalize_text, normalize_unicode, nlp

/Users/muhammadhadi/Desktop/NLP_Assignment_1/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
df = pd.read_csv("../data/Tweets.csv")
df = df[["text", "airline_sentiment"]].dropna(subset=["text"])

print("Total tweets:", len(df))
df.head()

Total tweets: 14640


,text,airline_sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


In [3]:
sample = "cafe\u0301"
result = normalize_unicode(sample)

print("Before:", repr(sample))
print("After:", repr(result))
print("NFC confirmed:", unicodedata.is_normalized("NFC", result))

Before: 'café'
After: 'café'
NFC confirmed: True


In [4]:
sample = "THE FLIGHTS ARE GOOD"
print("Before:", sample)
print("After:", sample.casefold())

Before: THE FLIGHTS ARE GOOD
After: the flights are good


In [5]:
sample = "I am not happy with this flight"
doc = nlp(sample)

kept = [
    token.text for token in doc
    if not token.is_stop or token.text.lower() in {"not", "no", "never"}
]

print("Before:", sample)
print("After:", " ".join(kept))

Before: I am not happy with this flight
After: not happy flight


In [16]:
words = ["flights", "passengers", "waiting", "delayed"]

for word in words:
    print(word, "->", nlp(word)[0].lemma_)

flights -> flight
passengers -> passenger
waiting -> wait
delayed -> delay


In [17]:
df["cleaned_text"] = df["text"].apply(normalize_text)

print("Original tweet:")
print(df.iloc[2]["text"])

print("\nNormalized tweet:")
print(df.iloc[2]["cleaned_text"])

Original tweet:
@VirginAmerica I didn't today... Must mean I need to take another trip!

Normalized tweet:
not today mean need trip


In [18]:
import re

def vocabulary(texts):
    words = []
    for text in texts:
        words.extend(re.findall(r"\b\w+\b", str(text).casefold()))
    return set(words)

before = vocabulary(df["text"])
after = vocabulary(df["cleaned_text"])

print("Vocabulary before:", len(before))
print("Vocabulary after:", len(after))
print("Words removed:", len(before - after))

Vocabulary before: 15089
Vocabulary after: 9958
Words removed: 6066
